# Running Falcon-X on gift-eval benchmark

**The following notebook is only intended to reproduce GIFT-Eval results using a GluonTS-style predictor interface. For practical usage, we recommend using the simpler interface of Falcon-X as described in the [Github repo](https://github.com/ant-intl/Falcon-TST/tree/main/falconx).**

Make sure you download the gift-eval benchmark and set the `GIFT-EVAL` environment variable correctly before running this notebook.

We will use the `Dataset` class to load the data and run the model. If you have not already please check out the [dataset.ipynb](./dataset.ipynb) notebook to learn more about the `Dataset` class. We are going to just run the model on two datasets for brevity. But feel free to run on any dataset by changing the `SHORT_DATASETS` and `MED_LONG_DATASETS` variables below.

Install our official Python SDK to access the Falcon-TST forecasting API—no local GPU or model deployment required:

``
pip install falcon-tst
``

In [ ]:
import json

from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# SHORT_DATASETS = "m4_yearly m4_quarterly m4_monthly m4_weekly m4_daily m4_hourly electricity/15T electricity/H electricity/D electricity/W solar/10T solar/H solar/D solar/W hospital covid_deaths us_births/D us_births/M us_births/W saugeenday/D saugeenday/M saugeenday/W temperature_rain_with_missing kdd_cup_2018_with_missing/H kdd_cup_2018_with_missing/D car_parts_with_missing restaurant hierarchical_sales/D hierarchical_sales/W LOOP_SEATTLE/5T LOOP_SEATTLE/H LOOP_SEATTLE/D SZ_TAXI/15T SZ_TAXI/H M_DENSE/H M_DENSE/D ett1/15T ett1/H ett1/D ett1/W ett2/15T ett2/H ett2/D ett2/W jena_weather/10T jena_weather/H jena_weather/D bitbrains_fast_storage/5T bitbrains_fast_storage/H bitbrains_rnd/5T bitbrains_rnd/H bizitobs_application bizitobs_service bizitobs_l2c/5T bizitobs_l2c/H"
SHORT_DATASETS = "us_births/D"

# MED_LONG_DATASETS = "electricity/15T electricity/H solar/10T solar/H kdd_cup_2018_with_missing/H LOOP_SEATTLE/5T LOOP_SEATTLE/H SZ_TAXI/15T M_DENSE/H ett1/15T ett1/H ett2/15T ett2/H jena_weather/10T jena_weather/H bitbrains_fast_storage/5T bitbrains_rnd/5T bizitobs_application bizitobs_service bizitobs_l2c/5T bizitobs_l2c/H"
MED_LONG_DATASETS = "solar/H"

# Get union of short and med_long datasets
all_datasets = list(set(SHORT_DATASETS.split() + MED_LONG_DATASETS.split()))

dataset_properties_map = json.load(open("dataset_properties.json"))

EVAL_QUANTILE_LEVELS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
TRAINING_QUANTILE_LEVELS = [0.01, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 0.99]

from gluonts.ev.metrics import (
    MAE,
    MAPE,
    MASE,
    MSE,
    MSIS,
    ND,
    NRMSE,
    RMSE,
    SMAPE,
    MeanWeightedSumQuantileLoss,
)

# Instantiate the metrics
metrics = [
    MSE(forecast_type="mean"),
    MSE(forecast_type=0.5),
    MAE(),
    MASE(),
    MAPE(),
    SMAPE(),
    MSIS(),
    RMSE(),
    NRMSE(),
    ND(),
    MeanWeightedSumQuantileLoss(
        quantile_levels=EVAL_QUANTILE_LEVELS
    ),
]

## Falcon-X inference batch construction

This section prepares the inputs required by Falcon-X during inference. Each task keeps only the `target` series, trims it to the configured context length, and batches the resulting rows as `context`.

For multivariate series, `group_ids` marks which target rows belong to the same original task, while `target_idx_ranges` is used to split the batched model output back into per-task forecasts.

In [ ]:
from typing import Iterator, Mapping, Sequence, TypeAlias
import logging
import sys

import numpy as np
import torch
from torch.utils.data import IterableDataset


TensorOrArray: TypeAlias = torch.Tensor | np.ndarray

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
    stream=sys.stdout,
    force=True,
)

logger = logging.getLogger("Falcon-X Predictor")
logger.setLevel(logging.INFO)


def left_pad_and_cat_2D(tensors: list[torch.Tensor]) -> torch.Tensor:
    """Left-pad contexts to a shared length and concatenate all target rows."""
    max_len = max(tensor.shape[-1] for tensor in tensors)
    padded = []
    for tensor in tensors:
        n_rows, length = tensor.shape
        if length < max_len:
            padding = torch.full((n_rows, max_len - length), fill_value=torch.nan, device=tensor.device)
            tensor = torch.cat([padding, tensor], dim=-1)
        padded.append(tensor)

    return torch.cat(padded, dim=0)


def validate_and_prepare_single_dict_task(task: Mapping[str, TensorOrArray], idx: int) -> torch.Tensor:
    """Validate a Falcon-X task and return its target rows as float32 context data."""
    keys = set(task.keys())
    if keys != {"target"}:
        raise ValueError(f"Element at index {idx} must contain only the required key 'target', but found {keys}")

    target = task["target"]
    if isinstance(target, np.ndarray):
        target = torch.from_numpy(target)
    if not isinstance(target, torch.Tensor):
        raise TypeError(f"Element at index {idx} has unsupported target type {type(target)}")
    if target.ndim not in (1, 2):
        raise ValueError(
            "`target` must be 1-d with shape (history_length,) or 2-d with shape "
            f"(n_variates, history_length). Found element at index {idx} with shape {tuple(target.shape)}."
        )

    history_length = target.shape[-1]
    return target.reshape(-1, history_length).to(dtype=torch.float32)


class TSBatchDataset(IterableDataset):
    """
    Create Falcon-X inference batches from target-only time-series tasks. Adapted from the Chronos-2 
    dataset processing approach (chronos-forecasting).
    """

    def __init__(
        self,
        inputs: Sequence[Mapping[str, TensorOrArray]],
        context_length: int,
        batch_size: int,
    ) -> None:
        super().__init__()
        self.tasks = TSBatchDataset._prepare_tasks(inputs)
        self.context_length = context_length
        self.batch_size = batch_size

    @staticmethod
    def _prepare_tasks(inputs: Sequence[Mapping[str, TensorOrArray]]) -> list[torch.Tensor]:
        tasks = [validate_and_prepare_single_dict_task(raw_task, idx) for idx, raw_task in enumerate(inputs)]
        if not tasks:
            raise ValueError("The dataset is empty. Please provide at least one time series.")
        return tasks

    def _construct_slice(self, task_idx: int) -> torch.Tensor:
        target = self.tasks[task_idx]
        if target.shape[-1] > self.context_length:
            return target[:, -self.context_length :]
        return target

    def _build_batch(self, task_indices: list[int]) -> dict[str, torch.Tensor]:
        batch_context_tensor_list = []
        batch_group_ids_list = []
        target_idx_ranges: list[tuple[int, int]] = []

        target_start_idx = 0
        for group_id, task_idx in enumerate(task_indices):
            task_context = self._construct_slice(task_idx)
            n_targets = task_context.shape[0]

            batch_context_tensor_list.append(task_context)
            batch_group_ids_list.append(torch.full((n_targets,), fill_value=group_id, dtype=torch.long))
            target_idx_ranges.append((target_start_idx, target_start_idx + n_targets))
            target_start_idx += n_targets

        return {
            "context": left_pad_and_cat_2D(batch_context_tensor_list),
            "group_ids": torch.cat(batch_group_ids_list, dim=0),
            "target_idx_ranges": torch.tensor(target_idx_ranges, dtype=torch.long),
        }

    def _generate_sequential_batches(self):
        task_idx = 0
        while task_idx < len(self.tasks):
            current_batch_size = 0
            task_indices = []

            while task_idx < len(self.tasks) and current_batch_size < self.batch_size:
                task_indices.append(task_idx)
                current_batch_size += self.tasks[task_idx].shape[0]
                task_idx += 1

            yield self._build_batch(task_indices)

    def __iter__(self) -> Iterator[dict[str, torch.Tensor]]:
        yield from self._generate_sequential_batches()

    @classmethod
    def convert_inputs(
        cls,
        inputs: Sequence[Mapping[str, TensorOrArray]],
        context_length: int,
        batch_size: int,
    ) -> "TSBatchDataset":
        return cls(inputs, context_length=context_length, batch_size=batch_size)


## Evaluation

We use the Falcon API client to generate quantile forecasts on the gift-eval benchmark datasets. For each test window, the notebook prepares the Falcon-X inputs, calls `client.quantile_predict`, converts the returned quantiles into GluonTS forecasts, and evaluates them with the gift-eval metrics. We follow the naming conventions explained in the [README](../README.md) file and store the results in a csv file called `all_results.csv` under the `results/Falcon-X` folder.

The first column in the csv file is the dataset config name which is a combination of the dataset name, frequency and the term:

```python
f"{dataset_name}/{freq}/{term}"
```


In [ ]:
all_datasets

In [ ]:
import logging


class WarningFilter(logging.Filter):
    def __init__(self, text_to_filter):
        super().__init__()
        self.text_to_filter = text_to_filter

    def filter(self, record):
        return self.text_to_filter not in record.getMessage()


gts_logger = logging.getLogger("gluonts.model.forecast")
gts_logger.addFilter(
    WarningFilter("The mean prediction is not stored in the forecast data")
)

In [ ]:
import os
import itertools
import pandas as pd

from gluonts.model import evaluate_forecasts
from gluonts.time_feature import get_seasonality

from gift_eval.data import Dataset
from falcon import FalconClient
from torch.utils.data import DataLoader
from einops import rearrange

client = FalconClient()
context_length = 8192
batch_size = 64

model_name = "ant-intl/Falcon-X"
output_dir = "../results/Falcon-X/"
os.makedirs(output_dir, exist_ok=True)

csv_file_path = os.path.join(output_dir, "all_results.csv")

def load_processed_datasets(csv_file_path: str) -> set[str]:
    if not os.path.exists(csv_file_path) or os.path.getsize(csv_file_path) == 0:
        return set()
    try:
        results_df = pd.read_csv(csv_file_path)
    except pd.errors.EmptyDataError:
        return set()
    if "dataset" not in results_df.columns:
        logger.warning("Existing result file %s has no dataset column; no datasets will be skipped.", csv_file_path)
        return set()
    return set(results_df["dataset"].dropna().astype(str))


def append_result_to_csv(result_row: dict, csv_file_path: str) -> None:
    write_header = not os.path.exists(csv_file_path) or os.path.getsize(csv_file_path) == 0
    pd.DataFrame([result_row]).to_csv(
        csv_file_path,
        mode="a",
        header=write_header,
        index=False,
    )


pretty_names = {
    "saugeenday": "saugeen",
    "temperature_rain_with_missing": "temperature_rain",
    "kdd_cup_2018_with_missing": "kdd_cup_2018",
    "car_parts_with_missing": "car_parts",
}
pretty_model_name = {
    "ant-intl/Falcon-X": "Falcon-X",
}

def test_per_window(
    context_length: int,
    prediction_length: int,
    quantile_levels: list[float],
    test_data_input: list[dict],
    batch_size: int,
    use_multivariate_data: bool
):
    """Run one rolling evaluation window through Falcon-X quantile prediction."""
    from gluonts.model.forecast import QuantileForecast

    training_quantile_levels = TRAINING_QUANTILE_LEVELS

    input_data = [{"target": item["target"]} for item in test_data_input]
    is_univariate_data = input_data[0]["target"].ndim == 1

    test_dataset = TSBatchDataset.convert_inputs(
        inputs=input_data,
        context_length=context_length,
        batch_size=batch_size
    )

    test_loader = DataLoader(
        test_dataset, batch_size=None, pin_memory=True,
        shuffle=False, drop_last=False,
    )

    all_predictions: list[torch.Tensor] = []

    for batch in test_loader:
        #   context:           [sum_variates_in_batch, ctx_len]
        #   group_ids:         [sum_variates_in_batch]
        #   target_idx_ranges: [n_tasks_in_batch, 2]

        context = batch["context"]
        group_ids = batch["group_ids"]
        target_idx_ranges = batch["target_idx_ranges"]
        context_np = context.numpy()
        group_ids_np = group_ids.numpy()

        if use_multivariate_data:
            result = client.quantile_predict(
                model_name = 'Falcon-X',
                context = context_np,
                prediction_length = prediction_length,
                group_ids = group_ids_np
            )
        else:
            result = client.quantile_predict(
                model_name = 'Falcon-X',
                context = context_np,
                prediction_length = prediction_length
            )

        preds = result['prob_prediction']
        preds = torch.Tensor(preds)
        for start, end in target_idx_ranges:
            all_predictions.append(preds[start:end].cpu())

    predictions = [rearrange(pred, "... q h -> ... h q") for pred in all_predictions]

    if set(quantile_levels).issubset(training_quantile_levels):
        quantile_indices = [training_quantile_levels.index(q) for q in quantile_levels]
        quantiles = [pred[..., quantile_indices] for pred in predictions]
    else:
        raise NotImplementedError("Quantile extrapolation not supported")

    quantiles = torch.stack(quantiles)                          # [N, variates, pred_len, Q]
    quantiles = quantiles.permute(0, 3, 2, 1).cpu().numpy()    # [N, Q, pred_len, variates]

    if is_univariate_data:
        quantiles = quantiles.squeeze(-1)  # [N, Q, pred_len]
        assert quantiles.shape[0] == len(input_data)
        assert quantiles.shape[1] == len(quantile_levels)
        assert quantiles.shape[2] == prediction_length
    logger.info("    quantiles shape: %s", quantiles.shape)

    forecasts = []
    for item, ts in zip(quantiles, test_data_input):
        forecast_start_date = ts["start"] + len(ts["target"])
        forecast = QuantileForecast(
            forecast_arrays=item,
            forecast_keys=list(map(str, quantile_levels)),
            start_date=forecast_start_date,
        )
        forecasts.append(forecast)

    return forecasts


def evaluate_on_dataset(
    ds_name: str,
    ds_term: str,
    batch_size: int,
    use_multivariate_data: bool = True,
):
    is_multivariate_source = (
        Dataset(
            name=ds_name,
            term=ds_term,
            to_univariate=False,
        ).target_dim
        > 1
    )
    to_univariate = is_multivariate_source and not use_multivariate_data

    dataset = Dataset(
        name=ds_name,
        term=ds_term,
        to_univariate=to_univariate
    )
    prediction_length = dataset.prediction_length
    logger.info("  Loaded dataset: %s (term=%s, pred_len=%d, to_univariate=%s)",
                    ds_name, ds_term, prediction_length, to_univariate)
    
    # Evaluate each rolling window independently.
    forecast_windows = []
    n_windows = dataset.test_data.windows
    for window_idx in range(n_windows):
        entries_window_k = list(itertools.islice(dataset.test_data.input, window_idx, None, n_windows))
        logger.info("  Window %d/%d: %d entries",
                        window_idx + 1, n_windows, len(entries_window_k))
        forecasts_window_k = list(test_per_window(
            context_length=context_length,
            prediction_length=prediction_length,
            quantile_levels=EVAL_QUANTILE_LEVELS,
            test_data_input=entries_window_k,
            batch_size=batch_size,
            use_multivariate_data=use_multivariate_data
        ))
        forecast_windows.append(forecasts_window_k)

    forecasts = [item for items in zip(*forecast_windows) for item in items] # interleave results again
    season_length = get_seasonality(dataset.freq)
    return evaluate_forecasts(
            forecasts,
            test_data=dataset.test_data,
            metrics=metrics,
            batch_size=1024,
            axis=None,
            mask_invalid_label=True,
            allow_nan_forecast=False,
            seasonality=season_length,
        ) \
    .reset_index(drop=True) \
    .to_dict(orient="records")


processed_datasets = load_processed_datasets(csv_file_path)
logger.info("Output CSV: %s | already processed: %d datasets", csv_file_path, len(processed_datasets))

for ds_num, ds_name in enumerate(all_datasets):
    ds_key = ds_name.split("/")[0]
    logger.info(f"Processing dataset: {ds_name} ({ds_num + 1} of {len(all_datasets)})")
    terms = ["short", "medium", "long"]
    for term in terms:
        if (term == "medium" or term == "long") and ds_name not in MED_LONG_DATASETS.split():
            continue

        if "/" in ds_name:
            ds_key = ds_name.split("/")[0]
            ds_freq = ds_name.split("/")[1]
            ds_key = ds_key.lower()
            ds_key = pretty_names.get(ds_key, ds_key)
        else:
            ds_key = ds_name.lower()
            ds_key = pretty_names.get(ds_key, ds_key)
            ds_freq = dataset_properties_map[ds_key]["frequency"]
        ds_config = f"{ds_key}/{ds_freq}/{term}"

        if ds_config in processed_datasets:
            logger.info("Skipping already processed dataset: %s", ds_config)
            continue

        logger.info(f"Generating forecasts for {ds_config}")
        result_metrics = evaluate_on_dataset(
            ds_name=ds_name,
            ds_term=term,
            batch_size=batch_size,
            use_multivariate_data=True,
        )
        result_metrics = {f"eval_metrics/{k}": v for k, v in result_metrics[0].items()}

        result_row = {
            "dataset": ds_config,
            "model": pretty_model_name.get(model_name, model_name),
            **result_metrics,
            "domain": dataset_properties_map[ds_key]["domain"],
            "num_variates": dataset_properties_map[ds_key]["num_variates"],
        }
        append_result_to_csv(result_row, csv_file_path)
        processed_datasets.add(ds_config)
        logger.info("Results for %s have been appended to %s.", ds_config, csv_file_path)
